# 1 Initialize the Database

All the code related to data management is in the `EnvironmentData` class. This makes life easier - for example: we can send the CatsUserID once and it becomes a class property. Then, when we call other operations we don't have to send this information again.

When you create a new instance of `EnvironmentData` and there is no database, it will pull historical data and initialize the database. 

In [1]:
# Clear prior data. 
import os, sys, shutil

# Add parent directory to Python path to import EnvironmentData.
sys.path.append(os.path.dirname(os.getcwd()))

# Get the EnvironmentData class.
from EnvironmentData import EnvironmentData 

# The project adds to existing data so we need to clear that data to get a solid test from scratch.
if os.path.exists('../data'):
    shutil.rmtree('../data')
    os.makedirs('../data')

# Initialize EnvironmentData. This will run the historical data pull.
envdt = EnvironmentData(
    #days_back = 365 * 2,
    days_back = 7,
    coris_enabled = True,
    licor_enabled = True,
    conserv_enabled = True, 
    testing = True,
    # Since we are running from the experiments/ folder, we need to tell the class to use the parent directory as home.
    home_directory = ".."
)

DEBUG: Enabled data sources: ['Coris', 'Conserv', 'LI-COR']


Gathering LI-COR readings: 100%|███████████████████████| 4/4 [00:01<00:00,  3.10it/s]


Detailed information is saved in the log:

In [2]:
# Detailed info is saved in the log.
with open('../data/EnvironmentData.log', 'r') as file:
    for line in file.read().splitlines()[:10]:
        print(line)

2025-11-15 13:55:04,434 - EnvironmentData - INFO - Initialized Conserv client with 5 customers
2025-11-15 13:55:04,436 - EnvironmentData - INFO - Enabled data sources: ['Coris', 'Conserv', 'LI-COR']
2025-11-15 13:55:04,436 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/cats/user/?ApiKey=XXXX&CatsUserID=XXXX
2025-11-15 13:55:06,317 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/sensor/historical/?ApiKey=XXXX&SensorID=21373&ReadingType=SensorReadingF&StartUTC=1762635304&EndUTC=1763240104&MinReadingSpacing=600&RequestedOutputFormat=raw
2025-11-15 13:55:08,164 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/sensor/historical/?ApiKey=XXXX&SensorID=21375&ReadingType=SensorReadingF&StartUTC=1762635304&EndUTC=1763240104&MinReadingSpacing=600&RequestedOutputFormat=raw
2025-11-15 13:55:10,017 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/sensor/historical/?ApiKey=XXXX&SensorID=21377&Readin

This saves our intermediate data to `data/sensor_readings.parquet`. 

Initially, we leave the data mostly as-is. We'll clean, add formatted dates, consolidate readings from the same device, etc. when moving to analytical steps, this preserves the source data so we can always change our mind later about how we decide to view it. 

However, at this point we are taking care to standardize the data format between different API sources. 

There are just a few columns because this is only historical data. We'll bring in current data shortly, and that will add more columns. 

In [3]:
import polars
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "Coris").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh
i64,i32,str,str,str,str,str,str,f32,f32
1762635304,1763240106,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.790001,null
1762635904,1763240106,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.790001,null
1762636504,1763240106,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.720001,null
1762637104,1763240106,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.68,null
1762637704,1763240106,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.669998,null


In [4]:
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "LI-COR").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh
i64,i32,str,str,str,str,str,str,f32,f32
1762635600,1763240224,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""",null,"""Temperature""",70.489525,null
1762636500,1763240224,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""",null,"""Temperature""",70.605362,null
1762637400,1763240224,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""",null,"""Temperature""",70.682579,null
1762638300,1763240224,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""",null,"""Temperature""",70.605362,null
1762639200,1763240224,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""",null,"""Temperature""",70.56675,null


In [5]:
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "Conserv").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh
i64,i32,str,str,str,str,str,str,f32,f32
1762635733,1763240118,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.907997,null
1762636639,1763240118,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.800003,null
1762637533,1763240118,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.674004,null
1762639333,1763240118,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.475998,null
1762640233,1763240118,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.349998,null


# 2 Get Current Readings

Now we can start gathering and appending readings. There is a function `get_current_readings` that is run throughout the day, every 10 minutes for example. This function creates a parquet file at `data/new-readings` with the UTC as a filename. At the end of the day, all these readings will be consolidated into the database. 

Here is a sample of the readings:

In [6]:
envdt.get_current_readings()

# Data is read into new-readings folder for consolidation at the end of the day.
import os
filename = os.listdir('../data/new-readings')[0]
print(filename)
polars.read_parquet('../data/new-readings/' + filename).sample(5)

Gathering Conserv current readings: 100%|█████████████| 1/1 [01:40<00:00, 100.73s/it]


1763240224.parquet


SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh
i64,i32,str,str,str,str,str,str,f32,f32
1763240181,1763240227,"""Conserv""","""conserv:333:c008789""","""BYCBA_030030111_N___""","""conserv:333:c008789:RH""","""BYCBA_030030111_N___ - RH""","""RH""",null,45.740002
1763239878,1763240227,"""Conserv""","""conserv:333:c009072""","""BBARCH0100001_______""","""conserv:333:c009072:Temperatur…","""BBARCH0100001_______ - Tempera…","""Temperature""",73.255997,null
1763239861,1763240227,"""Conserv""","""conserv:333:c009063""","""BYCBA_B100B06_______""","""conserv:333:c009063:RH""","""BYCBA_B100B06_______ - RH""","""RH""",null,42.93
1763239542,1763240227,"""Conserv""","""conserv:333:c009010""","""BYCBA_0200221_E_____""","""conserv:333:c009010:RH""","""BYCBA_0200221_E_____ - RH""","""RH""",null,44.880001
1763239555,1763240227,"""Conserv""","""conserv:333:c008914""","""BYCBA_030030104_S___""","""conserv:333:c008914:Temperatur…","""BYCBA_030030104_S___ - Tempera…","""Temperature""",72.68,null


# 3 Consolidate Readings

At the end of the day, new readings will be consolidated into the table. At the same time, the analytical tables will be generated. 

Analytical tables include:

* `device_readings.parquet`: Sensor readings reorganized to one row per Device and UTC, with measurements across columns vs measurements across rows.* 
* `sensors.parquet`: Information about the unique sensors. Includes information extracted from SensorName. Join this to Sensors during analysis to enhance with Building, Room, Direction, etc.
* `devices.parquet`: Information about unique devices. Includes information extracted from SensorName. 
* `utcs.parquet`: Information related to the UTC times in various datasets. Join to Sensors or Devices to enhance with Date, Time, Year, Hour, Weekday, etc.
* `sensor_readings_daily.parquet`: Example of sensor readings summarized to the daily level which reduces row count by 99.3% for even faster queries.
* `device_readings_daily.parquet`: Example of device readings summarized to the daily level which reduces row count by 99.3% for even faster queries. 

We fully re-generate analytical tables during each consolidation. The data is small enough that this is a fairly quick process, so re-running it in full each time will make it easy to ensure consistency as we expand and change the project. 

In [7]:
# To consolidate these into the database, run consolidate_readings.
envdt.consolidate_readings()

# New-readings files are gone now.
# They get deleted each day to confirm that they have been loaded into the database and prepare for the next consolidation.
print(os.listdir('../data/new-readings'))

C:\Users\super\Documents\arbaiza-consulting\environmental-sensor-poc\EnvironmentData.py:1210: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  historical["SensorID"].is_in(sensors["SensorID"]).not_()
C:\Users\super\Documents\arbaiza-consulting\environmental-sensor-poc\EnvironmentData.py:942: DeprecationWarning: `str.concat` is deprecated; use `str.join` instead. Note also that the default `delimiter` for `str.join` is an empty string, not a hyphen.
  polars.col("SensorID").str.concat(", ").alias("Sensors"),
C:\Users\super\Documents\arbaiza-consulting\environmental-sensor-poc\EnvironmentData.py:943: DeprecationWarning: `str.concat` is deprecated; use `str.join` instead. Note also that the default `delimiter` for `str.join` is an empty string, not a hyphen.
  polars.col("SensorName").str.concat(", ").alias("SensorN

ColumnNotFoundError: unable to find column "ConservCustomerID"; valid columns: ["SensorReadingUTC", "QueryUTC", "Source", "DeviceID", "DeviceName", "SensorID", "SensorName", "SensorType", "SensorReadingF", "SensorReadingRh", "SensorReadingUTC_SecondsFromPrior"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
Parquet SCAN [..\./data//sensor_readings.parquet]
PROJECT */11 COLUMNS

**^^ We want this to be empty** since we have consolidated new readings into the historical data. 

Once we are done working with data intake/processing, we close the class to release the file lock on the log file.

In [ ]:
# When done, close the connection to the logs. 
envdt.close()

Let's look at the data we have now:

In [ ]:
# Sensor Readings
# The first rows will be missing the extra fields like HexGatewayMac, etc.
#   I am pulling in some extra fields like DeviceID and DeviceName so we have that by historical. 
#   But some don't make sense to  backfill so they'll be null.
sensor_readings = polars.read_parquet('../data/sensor_readings.parquet')
sensor_readings.head()

In [ ]:
# Recent rows will have the full data, aside from nulls due to a sensor not providing a reading type.
sensor_readings.tail()

In [ ]:
# Device Readings.
device_readings = polars.read_parquet('../data/device_readings.parquet')
device_readings.head()

In [ ]:
# Sensors
sensors = polars.read_parquet('../data/sensors.parquet')
sensors.head()

In [ ]:
# Devices. 
devices = polars.read_parquet('../data/devices.parquet')
devices.head()

In [ ]:
# UTC Date/Time Info
utcs = polars.read_parquet('../data/utcs.parquet').head()
utcs.head()

In [ ]:
# Daily Sensor Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
sensor_readings_daily = polars.read_parquet('../data/sensor_readings_daily.parquet')
sensor_readings_daily.head()

In [ ]:
# Daily Device Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
device_readings_daily = polars.read_parquet('../data/device_readings_daily.parquet')
device_readings_daily.head()

In [ ]:
# Differentiate historical vs. cron readings by filtering on QueryUTC = NULL.
import duckdb
duckdb.sql("""
    SELECT *
    FROM read_parquet('../data/device_readings.parquet') 
    WHERE QueryUTC is null
    LIMIT 5
""").to_df()

Now you are ready to move onto analysis to get human-readable results (not indexed by UTC timestamps). See 2-examples-analysis.ipynb.